# Bias Bounty Mapping Equity Challenge - EDA & Baseline

This notebook explores the national strata table and computes baseline coverage gap features.

**Key insights to explore:**
1. Distribution of vulnerability indices across census tracts
2. Which *_covered flags are null (signal, not missing data)
3. Coverage patterns in the 4 focus regions
4. Source composition in Overture data (the `sources[]` column)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.helpers import set_all_seeds, setup_logging

set_all_seeds(42)
setup_logging('INFO')

# Font support
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf')
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 1. Load National Strata Table

In [ ]:
# Connect to DuckDB
conn = duckdb.connect()
conn.execute("INSTALL spatial; LOAD spatial;")
conn.execute("INSTALL httpfs; LOAD httpfs;")
conn.execute("SET s3_region='us-west-2';")
conn.execute("SET s3_access_key_id='';")
conn.execute("SET s3_secret_access_key='';")

# Try loading from local cache first, then S3
local_path = '../data/raw/national-strata-tract-table.parquet'
s3_path = 's3://us-west-2.opendata.source.coop/humane-intelligence/bias-bounty-mapping-equity-challenge/strata/national-strata-tract-table.parquet'

import os
if os.path.exists(local_path):
    print(f'Loading from local cache: {local_path}')
    conn.execute(f"CREATE OR REPLACE TABLE strata AS SELECT * FROM read_parquet('{local_path}')")
else:
    print(f'Loading from S3: {s3_path}')
    conn.execute(f"CREATE OR REPLACE TABLE strata AS SELECT * FROM read_parquet('{s3_path}')")

# Basic stats
count = conn.execute('SELECT COUNT(*) FROM strata').fetchone()[0]
cols = conn.execute("SELECT COUNT(*) FROM information_schema.columns WHERE table_name='strata'").fetchone()[0]
print(f'National strata: {count:,} tracts, {cols} columns')

## 2. Explore Column Types and Coverage

In [ ]:
# Get column info
col_info = conn.execute("""
    SELECT column_name, data_type 
    FROM information_schema.columns 
    WHERE table_name='strata'
    ORDER BY ordinal_position
""").df()

print(f'Total columns: {len(col_info)}')
print(f'\nColumn types:')
print(col_info['data_type'].value_counts())

# Find _covered columns
covered_cols = col_info[col_info['column_name'].str.endswith('_covered')]
print(f'\n*_covered columns: {len(covered_cols)}')
print(covered_cols['column_name'].tolist())

## 3. Analyze Null Patterns (NULL IS SIGNAL)

In [ ]:
# Compute null fractions for each _covered column
covered_names = covered_cols['column_name'].tolist()
null_stats = []

for col in covered_names:
    stats = conn.execute(f"""
        SELECT 
            SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) as null_count,
            SUM(CASE WHEN {col} = true THEN 1 ELSE 0 END) as true_count,
            SUM(CASE WHEN {col} = false THEN 1 ELSE 0 END) as false_count,
            COUNT(*) as total
        FROM strata
    """).fetchone()
    null_stats.append({
        'column': col,
        'null_pct': stats[0] / stats[3] * 100,
        'true_pct': stats[1] / stats[3] * 100,
        'false_pct': stats[2] / stats[3] * 100,
    })

null_df = pd.DataFrame(null_stats)
print('Coverage flag patterns:')
print(null_df.to_string())

# Key insight: high null % = CONUS-only data layer
conus_only = null_df[null_df['null_pct'] > 30]
print(f'\nCONUS-only layers (>{30}% null):')
print(conus_only[['column', 'null_pct']].to_string())

## 4. SVI and Vulnerability Distributions

In [ ]:
# Load to pandas for visualization
strata_df = conn.execute('SELECT * FROM strata').df()
print(f'Loaded: {strata_df.shape}')

# Find SVI columns
svi_cols = [c for c in strata_df.columns if 'svi' in c.lower()]
print(f'SVI columns: {svi_cols}')

# Plot SVI distribution
if svi_cols:
    fig, axes = plt.subplots(1, min(3, len(svi_cols)), figsize=(15, 4))
    if len(svi_cols) == 1:
        axes = [axes]
    for i, col in enumerate(svi_cols[:3]):
        strata_df[col].dropna().hist(bins=50, ax=axes[i], alpha=0.7)
        axes[i].set_title(col)
        axes[i].set_xlabel('Value')
    plt.tight_layout()
    plt.savefig('../docs/svi_distributions.png', dpi=150)
    plt.show()

## 5. Focus Region Breakdown

In [ ]:
# Analyze the 4 focus regions
focus_regions = {
    'Maricopa AZ': ('04', '013'),  # state 04, county 013
    'Eastern OK': ('40', None),
    'South-Central TX': ('48', None),
    'Northern CA': ('06', None),
}

if 'GEOID' in strata_df.columns:
    for name, (state, county) in focus_regions.items():
        if county:
            mask = strata_df['GEOID'].str.startswith(state + county)
        else:
            mask = strata_df['GEOID'].str.startswith(state)
        n = mask.sum()
        print(f'{name}: {n:,} tracts')
else:
    print('GEOID column not found in strata table')
    print('Available columns:', list(strata_df.columns[:20]))

## 6. Compute Null Flag Features

In [ ]:
from src.features.feature_engineering import NullFlagFeatures, VulnerabilityFeatures

# Null flags
flag_engineer = NullFlagFeatures()
flag_features = flag_engineer.compute_coverage_flags(strata_df)
print(f'Null flag features: {flag_features.shape}')
print(flag_features.head())

# Vulnerability features
vuln_engineer = VulnerabilityFeatures()
vuln_features = vuln_engineer.compute_vulnerability_features(strata_df)
print(f'\nVulnerability features: {vuln_features.shape}')